In [ ]:
# Choose the model, pipeline type and current execution to debug.
# The notebook reads that job's actual inputs, including cached producer paths.
# The historical examples below belong to namespace src1007cced-i1102a.
MODEL_ID = "usecase2"
PIPELINE_KIND = "testing"  # "testing" or "inference"
EXECUTION_ID = "qaq3ddxjfasy"
# Historical inference example: PIPELINE_KIND = "inference"; EXECUTION_ID = "brf4bz1lunno"
# Use an execution from the namespace in your notebooks/config.json.


In [ ]:
# Locate this checkout and inspect the existing Postprocess job; no pipeline is run.
# Show each input's physical producer separately from the requested execution.
# Keep the printed assignments usable on their own; the final cell exports a full VS Code block.
from pathlib import Path
import boto3
import json
import sys

REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
source_directory = str(REPOSITORY / "mlops-models/src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)
config = json.loads((REPOSITORY / "notebooks/config.json").read_text())
if PIPELINE_KIND not in {"testing", "inference"}:
    raise ValueError("Choose testing or inference.")
sm = boto3.client("sagemaker")
pipeline = sm.describe_pipeline(PipelineName=f"{config['namespace']}-{PIPELINE_KIND}")
execution_arn = pipeline["PipelineArn"] + "/execution/" + EXECUTION_ID

# Read Postprocess's job rather than guessing where cached producer outputs live.
lookup_arn = execution_arn
visited = set()
requested_steps = None
while True:
    if lookup_arn in visited:
        raise ValueError("Repeated cache reference: " + lookup_arn)
    visited.add(lookup_arn)
    steps = [step for page in sm.get_paginator("list_pipeline_execution_steps").paginate(
        PipelineExecutionArn=lookup_arn) for step in page["PipelineExecutionSteps"]]
    if requested_steps is None:
        requested_steps = {step["StepName"]: step for step in steps}
    step = next((step for step in steps if step["StepName"] == "Postprocess"), None)
    if step is None:
        raise ValueError("No Postprocess step found for " + lookup_arn)
    job_arn = step.get("Metadata", {}).get("ProcessingJob", {}).get("Arn")
    if job_arn:
        break
    cached_from = step.get("CacheHitResult", {}).get("SourcePipelineExecutionArn")
    if not cached_from:
        raise ValueError("Postprocess has no Processing job yet. Run it once before inspecting its inputs.")
    lookup_arn = cached_from

job = sm.describe_processing_job(ProcessingJobName=job_arn.rsplit("/", 1)[1])
environment = job.get("Environment", {})
if environment.get("MODEL_ID") != MODEL_ID:
    raise ValueError(f"This job belongs to {environment.get('MODEL_ID')!r}, not {MODEL_ID!r}.")
if environment.get("MLOPS_PREDICTION_FORMAT", "csv") != "csv":
    raise ValueError("This notebook uses Processing/CSV scoring, not native Batch Transform.")
mounts = {item["S3Input"]["LocalPath"].rstrip("/"): item["S3Input"]["S3Uri"].rstrip("/")
          for item in job["ProcessingInputs"] if "S3Input" in item}
DATA_URI = mounts["/opt/ml/processing/raw"] + "/data.csv"
SELECTION_URI = mounts["/opt/ml/processing/selection"] + "/selected_features.csv"
MODEL_URI = mounts["/opt/ml/processing/model"]
SCORE_OUTPUT_URI = mounts["/opt/ml/processing/input"] + "/"
REGION = sm.meta.region_name


# Everything below is a Python comment or assignment, so it is also copyable.
print(f"# Current {PIPELINE_KIND.upper()} execution: {EXECUTION_ID} | model: {MODEL_ID}")
print(f"# Postprocess job: {job_arn.rsplit('/', 1)[1]}")
postprocess_cache = requested_steps["Postprocess"].get("CacheHitResult", {}).get("SourcePipelineExecutionArn")
if postprocess_cache:
    print(f"# Postprocess itself reused from: {postprocess_cache.rsplit('/', 1)[-1]}")
for label, step_name, uri in (
    ("Raw data", "GetData", DATA_URI),
    ("Selected features", "Preprocess", SELECTION_URI),
    ("Score files", "Score", SCORE_OUTPUT_URI),
    ("Model archive", None, MODEL_URI),
):
    _, has_run, remainder = uri.rpartition("/runs/")
    producer = remainder.split("/", 1)[0] if has_run else "not encoded in URI"
    cache = requested_steps.get(step_name, {}).get("CacheHitResult", {})
    cached_from = cache.get("SourcePipelineExecutionArn")
    status = f"; cache hit from {cached_from.rsplit('/', 1)[-1]}" if cached_from else ""
    print(f"# {label}: stored under execution {producer}{status}")
print("# Raw data is Postprocess's input. Prepared data is Score's input.")
print("# The model's saved _execution.json identifies training origin after extraction.")
for name in ("MODEL_ID", "PIPELINE_KIND", "REGION", "DATA_URI", "SELECTION_URI",
             "MODEL_URI", "SCORE_OUTPUT_URI"):
    print(f"{name} = {globals()[name]!r}")


In [ ]:
# Download the raw rows, selection and model; copy every Score file with its relative path.
# Unpack the model and set the environment used by model code.
# Call the local checkout's entrypoint; keep results in a new local output directory.
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import json
import os
import tarfile
import boto3

work_directory = Path(mkdtemp(prefix="postprocess-"))
data_directory = work_directory / "raw"
selection_file = work_directory / "selection/selected_features.csv"
model_archive = work_directory / "model.tar.gz"
model_directory = work_directory / "model"
prediction_directory = work_directory / "input"
output_directory = work_directory / "output"
output_directory.mkdir()

# Local model reads and writable output; the entrypoint does not upload run reports.
os.environ.update({
    "MODEL_ID": MODEL_ID,
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_MODEL_ARTIFACT_URI": MODEL_URI,
    "MLOPS_WRITE_RUN_REPORT": "0",
})

# =============================================================================
# DOWNLOAD INPUTS
# =============================================================================
# DATA_URI is raw, not Preprocess/prepared/data.csv.
s3 = boto3.client("s3", region_name=REGION)
for uri, destination in (
    (DATA_URI, data_directory / "data.csv"),
    (SELECTION_URI, selection_file),
    (MODEL_URI, model_archive),
):
    print(f"Download: {uri}", flush=True)
    destination.parent.mkdir(parents=True, exist_ok=True)
    address = urlsplit(uri)
    s3.download_file(address.netloc, address.path.lstrip("/"), str(destination))

# Copy the whole Score folder, including messaging, importance files and nested output/.
address = urlsplit(SCORE_OUTPUT_URI)
prefix = address.path.lstrip("/").rstrip("/") + "/"
score_files = 0
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=address.netloc, Prefix=prefix):
    for item in page.get("Contents", []):
        key = item["Key"]
        if key.endswith("/"):
            continue
        destination = (prediction_directory / key[len(prefix):]).resolve()
        destination.relative_to(prediction_directory.resolve())
        destination.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(address.netloc, key, str(destination))
        score_files += 1
print(f"Score files: {score_files} | {prediction_directory}", flush=True)

# =============================================================================
# EXTRACT MODEL AND SHOW LOCAL PATHS
# =============================================================================
# The entrypoint receives an extracted directory, not model.tar.gz.
with tarfile.open(model_archive) as archive:
    archive.extractall(model_directory, filter="data")
training_record = model_directory / "_execution.json"
if training_record.is_file():
    training_origin = json.loads(training_record.read_text()).get("producer_execution_id") or "not recorded"
    print(f"Training producer recorded inside model: {training_origin}", flush=True)
else:
    print("Training producer not recorded inside this model archive.", flush=True)
print(json.dumps({
    "model_id": MODEL_ID,
    "data_directory": str(data_directory),
    "selection_file": str(selection_file),
    "input_path": str(prediction_directory),
    "model_dir": str(model_directory),
    "output_dir": str(output_directory),
}, indent=2), flush=True)

# =============================================================================
# NOTEBOOK ENTRYPOINT IMPORT
# =============================================================================
# The final export cell removes this section and calls the existing execute directly.
import importlib
if PIPELINE_KIND == "testing":
    from mlops_models.entrypoints import postprocess_test as postprocess
else:
    from mlops_models.entrypoints import postprocess_inference as postprocess
postprocess = importlib.reload(postprocess)
# =============================================================================
# EXECUTE POSTPROCESS
# =============================================================================
result_directory = postprocess.execute(
    model_id=MODEL_ID,
    input_path=prediction_directory,
    output_dir=output_directory,
    model_dir=model_directory,
    data_directory=data_directory,
    selection_file=selection_file,
)
print(result_directory)


In [ ]:
# Save this notebook before exporting so edits to the execution cell are included.
# Generate one block to paste at the bottom of the matching entrypoint, replacing its old main block.
# Use VS Code's "Python: Debug current model entrypoint" configuration; this cell does not execute the block.
from textwrap import indent

saved_notebook = json.loads((REPOSITORY / "notebooks/steps/postprocess.ipynb").read_text())
body = "".join(next(cell for cell in saved_notebook["cells"] if cell.get("id") == "execute")["source"])
import_marker = "# =============================================================================\n# NOTEBOOK ENTRYPOINT IMPORT\n"
execute_marker = "# =============================================================================\n# EXECUTE POSTPROCESS\n"
setup, found_import, remainder = body.partition(import_marker)
_, found_execute, execution = remainder.partition(execute_marker)
if not found_import or not found_execute:
    raise ValueError("Save the notebook with its import and execution dividers before exporting.")
body = setup + execute_marker + execution
body = body.replace("result_directory = postprocess.execute(", "result_directory = execute(", 1)
assignments = "\n".join(
    f"{name} = {globals()[name]!r}" for name in
    ("MODEL_ID", "PIPELINE_KIND", "REGION", "DATA_URI", "SELECTION_URI", "MODEL_URI", "SCORE_OUTPUT_URI")
)
entrypoint_name = "postprocess_test.py" if PIPELINE_KIND == "testing" else "postprocess_inference.py"
vscode_code = (
    f"# Paste at the bottom of {entrypoint_name}; replace the previous local main block.\n"
    'if __name__ == "__main__":\n'
    + indent(assignments + "\n\n" + body, "    ")
)
compile(vscode_code, entrypoint_name, "exec")
print(vscode_code)
